# Asking the bank's documentation

An employee of Orla Bank, a fictional bank, wants to put a new service into
production. The answer is spread over documents of three departments: the
technical guide says how to deploy, the release standard says who approves,
the compliance checklist says what must be checked first. Some documents
are old versions, and some disagree with each other.

This notebook follows that one question through the system, then asks
"what if ..." until every part of the assignment has appeared. The second
half is a reference: the other scenarios, learning, monitoring and the
evaluation numbers.

## The idea

```
 question
    |
    v
 classifier ---- reads the 8 closest chunks of the whole corpus,
    |            picks the departments (domains) to ask
    |
    +--------------------+--------------------+
    v                    v                    v
 technical agent     business agent     compliance agent
 searches only its   (in parallel)      (in parallel)
 own documents,
 writes claims, each
 citing its chunks
    |                    |                    |
    +--------------------+--------------------+
    v
 conflict check ---- a model finds claims that contradict each other;
    |                code picks the winner by fixed rules
    v
 synthesis --------- a model writes one answer citing [TEC-3] ...;
    |                code checks every cited id and lists the sources
    v
 answer, sources, warnings
```

Four principles carry the design:

1. **Code coordinates, models read text.** The steps above are plain Python
   in `rag_system/orchestrator.py`; a model is used only where text must be
   understood: routing, writing claims, spotting a contradiction, writing
   the answer.
2. **Every sentence has a source.** Agents write claims that cite chunk ids;
   code turns them into citations, and the answer may cite only documents
   the agents read.
3. **The winner of a conflict is decided by code, not chosen by a model.**
   Models do the reading: one finds the claims that contradict each other;
   for two documents of the same kind, a judge marks which other documents
   support each side. Code then applies fixed rules: a policy outranks a
   standard, a standard a guide, a guide a runbook; within one kind, the
   side with more supporting documents wins; a tie has no winner.
4. **"I don't know" is an answer.** An agent that finds nothing relevant
   says so; an answer is never filled with what the documents do not say.

In [1]:
import os
import sys
import time
from pathlib import Path

# Run from the repository root, so that .env, data/ and the package are found.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"

from rag_system.config import LLMProvider, Settings, build_embedder, build_llm
from rag_system.ingestion import chunk_document, load_documents
from rag_system.messages import Classification
from rag_system.monitoring import Monitor
from rag_system.orchestrator import Orchestrator
from rag_system.vector_store import InMemoryVectorStore

# .env chooses the model; without it everything runs on the offline stand-in.
settings = Settings()
model = build_llm(settings)
embedder = build_embedder(settings)
documents = load_documents(ROOT / "data")


def build_store():
    store = InMemoryVectorStore(embedder)
    for document in documents:
        store.add(chunk_document(document))
    return store


started = time.perf_counter()
store = build_store()
monitor = Monitor()
orchestrator = Orchestrator(store, model, monitor=monitor, timeout_seconds=60)

if settings.llm_provider is LLMProvider.OFFLINE:
    print("Language model: the OFFLINE stand-in (no key in .env). Answers are")
    print("templates built from the retrieved chunks, not model writing.")
else:
    print(f"Language model: {settings.llm_provider} ({getattr(model, 'model_name', model)})")
print(f"Embedding model: {store.model}")
print(f"Indexed {len(documents)} documents in {time.perf_counter() - started:.1f} s")

Language model: anthropic (claude-sonnet-5-5)
Embedding model: minishlab/potion-base-8M
Indexed 26 documents in 0.1 s


In [2]:
def show_answer(answer):
    """The answer text, its sources and any warnings."""
    print(answer.text)
    print("\nSources:")
    for source in answer.sources:
        section = f', "{source.section}"' if source.section else ""
        print(
            f"  [{source.doc_id}] {source.title}, v{source.version}, "
            f"{source.effective_date}{section}"
        )
    for warning in answer.warnings:
        print(f"Warning: {warning}")

## One question, step by step

The first test scenario of the assignment:

In [3]:
QUESTION = (
    "What's the process for deploying a new microservice and what "
    "compliance checks are needed?"
)
answer = await orchestrator.answer(QUESTION, Classification.INTERNAL)

**Step 1, the classifier.** It searches the whole corpus first and gives
the model the question with the 8 closest chunks and their departments; the
model decides which departments to ask and writes a sub-question for each.
Routing from evidence, not from department descriptions: on the first 12
questions, descriptions found 12 of the 18 departments needed, search 18 of
18.

In [4]:
for sub_query in answer.plan.sub_queries:
    history = " (old versions too)" if sub_query.include_history else ""
    print(f"{sub_query.domain:<11} {sub_query.text}{history}")

technical   What's the process for deploying a new microservice?
compliance  What compliance checks are needed for deploying a new microservice?


**Step 2, the domain agents**, in parallel. Each searches only its own
documents (embedding similarity and keyword search, merged), keeps the 8
best chunks plus up to 2 more that are close to the sub-question, and writes
short claims that cite those chunks. The agents never talk to each other.

In [5]:
for agent_answer in answer.agent_answers:
    print(
        f"{agent_answer.domain}: {agent_answer.status}, read "
        f"{len(agent_answer.chunks_read)} chunks, wrote "
        f"{len(agent_answer.claims)} claims; for example:"
    )
    for claim in agent_answer.claims[:2]:
        cited = ", ".join(c.chunk_id for c in claim.citations)
        print(f"    {claim.text}  <- {cited}")

technical: answered, read 8 chunks, wrote 14 claims; for example:
    Before coding, register the service in the service catalog with an owning team, a short description and the highest data classification it will handle (CMP-1).  <- TEC-1#2
    Create the repository from the platform service template. It already includes the CI pipeline, a health endpoint, structured logging and the metrics required by the Observability Standard (TEC-6).  <- TEC-1#2
compliance: answered, read 9 chunks, wrote 19 claims; for example:
    A new service or connector must pass the Pre-Production Compliance Checklist before its first production release.  <- CMP-4#1
    The completed checklist is attached to the release request (Production Release Standard, TEC-3).  <- CMP-4#1


**Step 3, the conflict check.** A model reads all claims and names the
pairs that cannot both be true; code settles each pair (next section).

In [6]:
print(f"{len(answer.conflicts)} conflicts found")
for conflict in answer.conflicts:
    print(f"  {conflict.reason}")

0 conflicts found


**Step 4, the synthesis.** A model writes one answer from the claims and
cites document ids in brackets. Code checks that every cited id belongs to
a claim it was given (otherwise the model is asked again) and builds the
source list from the claims' citations, so no source is written by the
model.

In [7]:
show_answer(answer)

## Deployment process

1. **Register the service** in the service catalog before coding. Give an owning team, a short description and the highest data classification it will handle [TEC-1].
2. **Create the repository** from the platform service template. It already includes the CI pipeline, a health endpoint, structured logging and the metrics required by the Observability Standard [TEC-1]. If the template doesn't fit, ask in the Platform Engineering channel and do not fork the pipeline [TEC-1].
3. **Core banking access:** do not call core banking systems directly. Use an existing connector from the connector registry, or build one as described in the connector guidance [TEC-1].
4. **Pipeline on every push:** unit tests, SAST and dependency vulnerability scan, container image build and signing, then automatic deployment to `dev` [TEC-1]. A high or critical finding stops the pipeline [TEC-1].
5. **Promote to `test` and `staging`** with the pipeline button. Staging uses masked copies of 

## What if the documents disagree?

The API troubleshooting runbook keeps debug logs for 30 days; the data
retention policy says 90 days. The agents write both as claims, the
detector pairs them, and the rule decides: a policy outranks a runbook.

In [8]:
answer = await orchestrator.answer(
    "How long can we keep the troubleshooting logs of an API?",
    Classification.INTERNAL,
)
for conflict in answer.conflicts:
    print(f"{conflict.decision}: {conflict.reason}")
    print(f"  A: {conflict.claim_a.text}")
    print(f"  B: {conflict.claim_b.text}")
print()
show_answer(answer)

kind: CMP-3 (policy) overrules TEC-4 (runbook)
  A: Debug logs, which are turned on for troubleshooting an API, are kept for 30 days and then deleted.
  B: API request logs, including debug logs, are kept for 90 days and then deleted.

Troubleshooting logs of an API, including debug logs, can be kept for 90 days and are then deleted [CMP-3]. Application logs follow the same 90-day period [CMP-3]. The Observability Standard says general log retention follows the Data Retention Policy (CMP-3) [TEC-6]. The 90 days cover the longest regular investigation: monthly reconciliation plus the time to open and close a case [CMP-3].

**Settled conflict:** One claim said debug logs turned on for troubleshooting an API are kept for 30 days and then deleted [TEC-4]. It is overruled. The policy (CMP-3) takes precedence over the runbook (TEC-4), so the 90-day period applies [CMP-3].

**Conditions for handling the logs:**
- Debug logging for the API should be turned on for at most 24 hours [TEC-4]. This

## What if a rule changes tomorrow?

A knowledge update is one `store.add()` of the new document. The new
version names the old one in `supersedes`; the store keeps the old version
for history but searches only the version in force. Here a made-up
version 3 of the release standard asks for three approvers.

In [9]:
from datetime import date

from rag_system.ingestion import Document

RELEASE_QUESTION = "Is one sign-off enough to ship a change to production?"
before = await orchestrator.answer(RELEASE_QUESTION, Classification.INTERNAL)

version_3 = Document(
    id="TEC-10",
    title="Production Release Standard",
    domain="technical",
    kind="standard",
    version=3,
    effective_date=date(2026, 11, 1),
    supersedes="TEC-3",
    classification="internal",
    owner="Platform Engineering",
    body="""# Production Release Standard (version 3)

This standard replaces version 2 (TEC-3) from 1 November 2026.

## Approvals

A production release needs **three approvers**: a senior engineer from the
team that owns the service, an engineer from Platform Engineering and an
engineer from Information Security. None of them may be the author.
""",
)
updated_store = build_store()  # a copy, so the rest of the notebook is unchanged
updated_store.add(chunk_document(version_3))
updated = Orchestrator(updated_store, model, timeout_seconds=60)
after = await updated.answer(RELEASE_QUESTION, Classification.INTERNAL)

for label, result in [("Before", before), ("After", after)]:
    cited = sorted({s.doc_id for s in result.sources})
    print(f"{label} the update, sources {cited}:")
    print(result.text, "\n")

Before the update, sources ['TEC-1', 'TEC-3']:
No, one sign-off is not enough.

- A production release needs two approvers: a senior engineer from the team that owns the service, and an engineer from Platform Engineering [TEC-3].
- Neither approver may be the author of the change [TEC-3].
- If the change touches confidential data, it also needs a separate four-eyes review under CMP-6, in addition to the release approvals [TEC-3].
- A release outside the window (Tuesday to Thursday, 09:00 to 15:00) is an emergency change. It needs the same two approvers plus the approval of the Platform Engineering lead [TEC-3]. The on-call Platform Engineering engineer may act as the second approver [TEC-3].
- The deployment guide tells you to get the approvals required by the Production Release Standard before going to production [TEC-1].

The claims do not say whether the on-call engineer can stand in for the Platform Engineering lead's approval. They only say the on-call engineer may act as the seco

## What if people say an answer was not useful?

The learning mechanism is simulated: a user rates every source of an
answer as useful or not (here by whether its document is one the gold set
expects, with 10% of the ratings wrong on purpose). A rating is stored with
the question it was given for and counts only for a similar question
(cosine similarity of at least 0.6), so it cannot leak into unrelated
questions. A chunk rated not useful sinks in the order and loses its
guaranteed place among the 8 best, so the agents stop reading it. (A model
cites only what it uses, so the number of sources moves less than the
chunks the agents read; the gold-set table in the reference counts both.)

In [10]:
import random

from rag_system.learning import ChunkWeights, SimulatedUser

LEARN_QUESTION = (
    "How do I troubleshoot API performance issues while following our "
    "security policies?"
)
EXPECTED = {"TEC-4", "TEC-6", "CMP-5"}

weights = ChunkWeights(embedder)
learner = Orchestrator(store, model, weights=weights, timeout_seconds=60)
user = SimulatedUser(random.Random(42), wrong_share=0.1)


def off_target(result):
    return sorted({s.doc_id for s in result.sources} - EXPECTED)


first = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)
for _ in range(3):
    rated = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)
    weights.apply(LEARN_QUESTION, user.rate(rated, EXPECTED))
last = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)

def read(result):
    return {chunk_id for a in result.agent_answers for chunk_id in a.chunks_read}


learned = weights.for_question(LEARN_QUESTION)
rated_down = sorted({r.chunk_id for r in weights.history if learned.weight(r.chunk_id) < 1})
rated_up = sorted({r.chunk_id for r in weights.history if learned.weight(r.chunk_id) > 1})
print(f"Ratings given: {len(weights.history)}")
print(f"Sources from unexpected documents, before: {off_target(first)}, after: {off_target(last)}")
print("\nChunks rated not useful (weight below 1): read by the agents before and after")
for chunk_id in rated_down:
    print(f"  {chunk_id:<9} weight {learned.weight(chunk_id):.2f}  "
          f"before: {'read' if chunk_id in read(first) else '-':<5} "
          f"after: {'read' if chunk_id in read(last) else '-'}")
print(f"\nChunks rated useful (weight above 1): {len(rated_up)}, "
      f"read after feedback: {len(set(rated_up) & read(last))}")

Ratings given: 31
Sources from unexpected documents, before: ['TEC-9'], after: ['TEC-5']

Chunks rated not useful (weight below 1): read by the agents before and after
  CMP-2#6   weight 0.80  before: read  after: -
  CMP-4#2   weight 0.80  before: read  after: -
  CMP-5#6   weight 0.80  before: read  after: -
  TEC-9#3   weight 0.64  before: read  after: -

Chunks rated useful (weight above 1): 8, read after feedback: 8


## What if no document answers?

Meeting rooms are not in the platform's documentation. The classifier may
ask no department at all, and an agent that is asked should report
"nothing relevant"; either way the answer says so.
(With the offline stand-in the agents answer anyway: its 0.4 similarity bar
is a known gap, see the evaluation below.)

In [11]:
answer = await orchestrator.answer(
    "Can you tell me which meeting rooms are available next Wednesday for a "
    "workshop with the process owners?",
    Classification.INTERNAL,
)
print(f"Departments asked: {[a.domain.value for a in answer.agent_answers]}")
for agent_answer in answer.agent_answers:
    print(f"{agent_answer.domain}: {agent_answer.status}")
print(answer.text)

Departments asked: []
The documentation holds nothing that answers this question.


## What if a part breaks?

An empty question is refused before any search. When one part fails after
the question is accepted, the answer keeps what the others found and says
what is missing. Here the compliance part of the index is down.

In [12]:
try:
    await orchestrator.answer("   ", Classification.INTERNAL)
except ValueError as error:
    print(f"Empty question: ValueError({error})")


class ComplianceIndexDown:
    """The real store, except that a search of compliance documents fails."""

    def __init__(self, store):
        self._store = store
        self.model = store.model

    def add(self, chunks):
        self._store.add(chunks)

    def search(self, query, domain, clearance, limit, **options):
        if domain == "compliance":
            raise ConnectionError("compliance index unavailable")
        return self._store.search(query, domain, clearance, limit, **options)


broken = Orchestrator(ComplianceIndexDown(store), model, timeout_seconds=60)
answer = await broken.answer(QUESTION, Classification.INTERNAL)
for agent_answer in answer.agent_answers:
    error = f" ({agent_answer.error})" if agent_answer.error else ""
    print(f"{agent_answer.domain}: {agent_answer.status}{error}")
print()
show_answer(answer)

Empty question: ValueError(the question is empty)


technical: answered
compliance: failed (ConnectionError: compliance index unavailable)

**Deployment process for a new microservice**

1. **Register the service.** Add it to the service catalog with an owning team, a short description and the highest data classification it will handle (CMP-1). [TEC-1]
2. **Create the repository.** Use the platform service template. It already includes the CI pipeline, a health endpoint, structured logging and the metrics required by the Observability Standard (TEC-6). [TEC-1] If the template doesn't fit, ask in the Platform Engineering channel and don't fork the pipeline. [TEC-1]
3. **Connect to core banking systems, if needed.** Don't call them directly. Use an existing connector from the connector registry, or build one as described in TEC-5. [TEC-1]
4. **Push code.** Every push runs the same pipeline: unit tests, SAST and a dependency vulnerability scan, container image build and signing, then automatic deployment to `dev`. [TEC-1]
5. **Promote to `

---

# Reference

## The other two test scenarios

In [13]:
for question in [
    "How do I troubleshoot API performance issues while following our security policies?",
    "What business approvals are required for implementing a new data processing workflow?",
]:
    answer = await orchestrator.answer(question, Classification.INTERNAL)
    print(f"Q: {question}\n")
    show_answer(answer)
    print("\n" + "-" * 72 + "\n")

Q: How do I troubleshoot API performance issues while following our security policies?

**Troubleshooting API performance within security policy**

Use the API Performance Troubleshooting Runbook when a platform API is slower than its objective of p95 latency below 800 ms. The alert fires when p95 stays above 800 ms for 10 minutes [TEC-4].

**Step 1: Confirm and scope**
- Open the API's latency dashboard and check p95 and p99 for the last hour [TEC-4].
- Check whether all endpoints are slow or only some. One slow endpoint usually means one slow dependency [TEC-4].
- Check the error rate. If it is above 2%, treat the case as an incident and follow the Platform Incident Response Runbook [TEC-4].

**Step 2: Look for a recent change**
- List the releases of the last 24 hours for the API and its dependencies in the change log [TEC-4].
- If a release matches the start of the slowdown, roll it back and watch the dashboard for 15 minutes [TEC-4].

**Step 3: Check dependencies**
- Open the trac

Q: What business approvals are required for implementing a new data processing workflow?

The business approvals required depend on the data the workflow uses.

**Approval tiers**
- A process that uses only public or internal data needs approval from the head of the requesting department only [BUS-2].
- A process that handles customer data (confidential under CMP-1) needs approval from the head of the requesting department and from the Process Board [BUS-2].
- A process that sends any data to the cloud language model needs those approvals and also a sign-off from the Data Protection Office (CMP-2) [BUS-2].

**Change from earlier policy**
- The department head is no longer the only approver. Version 1 of the policy required only the department head [BUS-2]. The tiers above therefore apply, and the department-head-only route now covers only processes using public or internal data [BUS-2].

**Process Board timing**
- The Process Board meets every two weeks [BUS-2].
- A request must reach 

## A conflict with no winner

BUS-6 says the process analyst reads the audit records of the workflows
they built; CMP-7 leaves analysts out of the read-access list. Both are
standards, and no other document supports either side, so the answer shows
both and names the owners who must settle it.

In [14]:
answer = await orchestrator.answer(
    "Can the analyst who built a workflow look at its audit records?",
    Classification.INTERNAL,
)
for conflict in answer.conflicts:
    print(f"{conflict.decision}: {conflict.reason}")
print()
show_answer(answer)

no_winner: BUS-6 and CMP-7 are both standards and no other document supports each; ask the owners: Process Excellence Office; Compliance, Internal Audit
no_winner: BUS-6 and CMP-7 are both standards and no other document supports each; ask the owners: Process Excellence Office; Compliance, Internal Audit

This question has no settled answer, because two standards disagree and neither outranks the other.

**Business position:** The process analyst reads the audit records of the workflows they assembled [BUS-6]. The stated purpose is to check the first runs after go-live [BUS-6].

**Compliance position:** Read access to audit records is limited to Compliance, Internal Audit, the Data Protection Office and, for their own workflows, process owners [CMP-7]. Compliance's documents do not say that a workflow's builder or analyst has read access as such. Access depends on belonging to one of those groups, or on being the process owner of that workflow [CMP-7].

**Why this is unresolved:** BUS-

## A question about the past

Superseded versions stay in the index but are searched only when the
classifier's model marks the question as one about history.

In [15]:
answer = await orchestrator.answer(
    "How many approvers did a production release need in 2025?",
    Classification.INTERNAL,
)
show_answer(answer)

In 2025, a production release needed one approver. Version 1 of the Production Release Standard applied that year [TEC-2, TEC-3].

- The approver had to be a senior engineer from the team that owns the service, and could not be the author of the change [TEC-2].
- A release outside the release window counted as an emergency change under version 1. It needed the Platform Engineering lead's approval in addition to the normal approver [TEC-2].

The two-approver rule does not apply to 2025. Version 2 replaces version 1 from 1 February 2026. It requires a senior engineer from the owning team and an engineer from Platform Engineering [TEC-3].

Sources:
  [TEC-2] Production Release Standard, v1, 2025-01-15, "Approvals"
  [TEC-2] Production Release Standard, v1, 2025-01-15, "Release windows"
  [TEC-3] Production Release Standard, v2, 2026-02-01
  [TEC-3] Production Release Standard, v2, 2026-02-01, "Approvals"
  [TEC-3] Production Release Standard, v2, 2026-02-01, "What changed from version 1"


## Learning over the gold set

The 19 tuning questions answered before and after three rounds of
simulated feedback. This table always runs on the offline stand-in: it
answers the 19 questions eight times, and the weights act on retrieval,
which is the same code in both modes. "Off-target" counts sources from
documents the question does not expect. The first row is circular (the
simulated user rates by the same expected documents that the row counts);
the paraphrases and the questions that were never rated are the honest
rows.

In [16]:
import yaml

from rag_system.offline_model import offline_model

gold = yaml.safe_load((ROOT / "tests/evals/questions.yaml").read_text())
tuning = [q for q in gold if not q.get("heldout")]


async def found_and_off_target(system, questions, field="question"):
    found = off = 0
    for q in questions:
        result = await system.answer(q[field], Classification(q.get("clearance", "internal")))
        expected = set(q["expected_documents"])
        found += len(expected & {s.doc_id for s in result.sources})
        off += sum(s.doc_id not in expected for s in result.sources)
    return found, off


async def feedback(system, table_weights, questions):
    rater = SimulatedUser(random.Random(42), wrong_share=0.1)
    for _ in range(3):
        for q in questions:
            result = await system.answer(q["question"], Classification(q.get("clearance", "internal")))
            table_weights.apply(q["question"], rater.rate(result, set(q["expected_documents"])))


rows = []
table_weights = ChunkWeights(embedder)
offline_learner = Orchestrator(store, offline_model(), weights=table_weights)
same_before = await found_and_off_target(offline_learner, tuning)
para_before = await found_and_off_target(offline_learner, tuning, "paraphrase")
await feedback(offline_learner, table_weights, tuning)
rows.append(("same questions", same_before, await found_and_off_target(offline_learner, tuning)))
rows.append(("paraphrased", para_before, await found_and_off_target(offline_learner, tuning, "paraphrase")))

table_weights = ChunkWeights(embedder)
offline_learner = Orchestrator(store, offline_model(), weights=table_weights)
not_rated = tuning[1::2]
before_nr = await found_and_off_target(offline_learner, not_rated)
await feedback(offline_learner, table_weights, tuning[0::2])
rows.append(("never rated", before_nr, await found_and_off_target(offline_learner, not_rated)))

print(f"{'':<16}{'found before':>13}{'off-target':>11}{'found after':>13}{'off-target':>11}")
for name, (f0, o0), (f1, o1) in rows:
    print(f"{name:<16}{f0:>13}{o0:>11}{f1:>13}{o1:>11}")

                 found before off-target  found after off-target
same questions             29        210           28         11
paraphrased                28        284           28        182
never rated                14         64           14         64


## Monitoring

Every answer of the orchestrator above was recorded: outcome, duration and,
per agent, its status and time. In production the same records would go to
OpenTelemetry; PydanticAI already instruments the model calls.

In [17]:
print(monitor.report())

questions: 8, successful: 88%

agent        asked  answered  nothing  failed  seconds  useful
technical        5      100%       0%      0%     4.93       -
business         2      100%       0%      0%     4.64       -
compliance       4      100%       0%      0%     5.35       -


## Evaluation

The gold set (`tests/evals/questions.yaml`) has 19 tuning questions, on
which the thresholds were set, and 35 held-out questions, written later by
other models from the document titles alone and run once with the
thresholds frozen. The offline measures below run here, with the same code
the tests use (`tests/evals/measures.py`).

In [18]:
from tests.evals.measures import (
    classifier_counts,
    document_ranks,
    fact_counts,
    store_counts,
)

heldout = [q for q in gold if q.get("heldout")]
print(f"{'offline measure':<34}{'tuning':>10}{'held-out':>10}")
results = {}
for name, questions in [("tuning", tuning), ("held-out", heldout)]:
    found, extra, _, _ = await classifier_counts(store, questions)
    expected_domains = sum(len(q["expected_domains"]) for q in questions)
    in_top_5, documents, _ = store_counts(store, questions)
    ranks = document_ranks(store, questions)
    facts, all_facts, _ = await fact_counts(store, questions)
    results[name] = [
        f"{found}/{expected_domains}",
        f"{extra / len(questions):.1f}",
        f"{in_top_5 / documents:.0%}",
        f"{sum(1 / r for r in ranks) / len(ranks):.2f}",
        f"{facts / all_facts:.0%}",
    ]
labels = [
    "classifier: domains found",
    "classifier: extra per question",
    "document in its domain's top 5",
    "mean reciprocal rank (random 0.34)",
    "facts in the chunks kept",
]
for i, label in enumerate(labels):
    print(f"{label:<34}{results['tuning'][i]:>10}{results['held-out'][i]:>10}")

offline measure                       tuning  held-out


classifier: domains found              26/26     38/38
classifier: extra per question           0.7       1.1
document in its domain's top 5           91%       68%
mean reciprocal rank (random 0.34)      0.75      0.59
facts in the chunks kept                 92%       65%


With the language model (`tests/evals/test_llm.py`, claude-sonnet-5-5,
2026-10-30; run with `uv run pytest -m eval` and a key in `.env`):

| measure | tuning | held-out | common target |
|---|---|---|---|
| facts in the chunks the model read (context recall) | 92% | 68-70% | 75-80% |
| facts stated in the answer (judge model) | 88% | 73-76% | |
| statements supported by the sources (faithfulness) | 97% overall | | 90-95% in regulated work |
| questions no document answers, ended as "nothing relevant" | | 6 of 7 | |
| conflicts found and decided as expected | 3-5 of 5 per run | | |

The gap between tuning and held-out is in retrieval: on new wording the
chunk with the answer often ranks below the first 8 of its department.
The local embedding model (potion-base-8M, chosen so the demo runs without
a key) is the main limit; production would use Azure OpenAI embeddings.